# Reproducing Le Merre, Heining et al. (2026) from dandiset 001260

**Paper.** Le Merre P, Heining K, Slashcheva M, Jung F, Moysiadou E, Guyon N, Yahya R, Park H,
Wernstal F, Carlén M. A prefrontal cortex map based on single-neuron activity. *Nature
Neuroscience* (2026). [doi:10.1038/s41593-025-02190-z](https://doi.org/10.1038/s41593-025-02190-z).
The claims file (`claims/001260/10.1038-s41593-025-02190-z.yaml`) was written from the version of
record (publisher HTML: main text, legends, Methods and Extended Data legends); the Supplementary
Information was not available to it.

**Dandiset.** [DANDI:001260](https://dandiarchive.org/dandiset/001260/0.250911.0744), version
0.250911.0744. The paper cites no version. This is the only published version, and the draft
has the same 74 assets and size, so it is pinned.

**Other data.** The paper deposits its dataset KI in two dandisets, 000473 and 001260 (Data
availability). Dandiset 000473 (version 0.230417.1502, the only published one; its metadata
names another paper) holds 25 more recordings; it is read here only for the unit counts of
`dataset-ki-counts` and `fig1h-waveform-classes`. From the paper's Zenodo record
[10.5281/zenodo.14205018](https://doi.org/10.5281/zenodo.14205018) this attempt reads the
Harris et al. hierarchy table (`hierarchy_summary_CreConf.xlsx`), the per-unit metrics files
(`metrics_files.zip`, for each unit's flatmap position), the flatmap polygons
(`flatmaps.zip`) and the Gao et al. ROI hierarchy table (`ROI_Gao_hierarchy_index.csv`).

**Methods.** The helper module `pfcki.py` says where every parameter comes from. Units are not
filtered by a quality label, as the files carry none (see the data check); every analysis
keeps units with at least 12 active spontaneous epochs of at least 6 spikes.

**Claims.** Tested here: the claims on dataset KI's spontaneous activity (Figs. 1 and 2,
Extended Data Figs. 1, 3 and 5), the Harris hierarchy correlations (Fig. 3a,c, Supplementary
Table 5) with the Harris table, the PFC flatmap claims (Fig. 4) through the Zenodo flatmap
positions after a link claim, and the tone-response claims on dataset KI (Fig. 5b to 5f, the
dataset part of 5i, 5j, Extended Data Fig. 9e). Blocked: every claim on dataset IBL (Figs. 2i,
3b, 5g, 5h, 5k), not attempted in this run's time, and `methods-settings-no-difference`, whose
test the paper does not name. Many enrichment claims are blocked by the claims file's own rule
because their outcome depends on which retrained cluster takes which category number (G2).
The table of outcomes is in the summary.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pfcki as P

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

t_start = time.time()
assets = P.list_assets()
recs = P.classify_epochs(P.load_all(assets))
U = P.unit_table(recs)
print(f"dandiset {P.DANDISET} version {P.VERSION}: {len(assets)} assets, {len(recs)} files read, "
      f"{len(U)} units, in {time.time() - t_start:.0f} s")
assets473 = P.list_assets(P.OTHER_DANDISET, P.OTHER_VERSION)
recs473 = P.classify_epochs(P.load_all(assets473))
U473 = P.unit_table(recs473)
print(f"dandiset {P.OTHER_DANDISET} version {P.OTHER_VERSION}: {len(recs473)} files, {len(U473)} units, "
      f"total {time.time() - t_start:.0f} s")

## Data check

In [ ]:
meta = P.version_metadata()
print("Archive summary:", meta["assetsSummary"]["numberOfFiles"], "files,",
      meta["assetsSummary"]["numberOfSubjects"], "subjects")
info = pd.DataFrame([{"session": r["session"], "subject": r["subject"], "setting": r["setting"],
                      "units": len(r["units"]), "onsets": len(r["onsets"]), "cortical_units": r["n_cortical"],
                      "duration_min": round((r["t1"] - r["t0"]) / 60, 1), "blocks": r["blockmode"],
                      "description": r["desc"][:58]} for r in recs])
print(info.to_string(index=False))
print("\nFiles, subjects and units by behavioral setting (setting from the session description and keywords):")
print(info.groupby("setting").agg(files=("session", "size"), subjects=("subject", "nunique"), units=("units", "sum")).to_string())
print("Distinct subjects:", info.subject.nunique(), "| sessions (subject and date):",
      info.session.str.split("-").str[0].nunique())

The archive counts 43 subjects, as the subject ids of two mice recorded on two days carry the
suffixes `-1` and `-2` (518804 and 560171); without the suffixes the files hold 41 mice, which with
the 25 mice of 000473 make the paper's 66. The
setting read from each file's description agrees with the setting the paper's code assigns to
the same recording (`config/datatables/allrecs_allprobes.xlsx`) for every file: Aversion 6,
Detection 15, Context 14, working memory ("Attention" in the code) 15, Opto 4, Passive 20.
Five auditory-cortex probes are described as Neuronexus recordings; the paper pools them as
probes of dataset KI.

In [ ]:
def rng_str(x):
    x = np.asarray(x, float)
    return f"{np.nanmin(x):.3g} to {np.nanmax(x):.3g} (median {np.nanmedian(x):.3g})"

print("Peak_to_Trough (ms):", rng_str(U.Peak_to_Trough if "Peak_to_Trough" in U else U.ptt))
print("quality column 0, 'ISI violation':", rng_str(U.q_isi), "| fraction < 0.1:", round((U.q_isi < 0.1).mean(), 3))
print("quality column 1, 'false positive rate':", rng_str(U.q_fp), "| fraction < 0.1:", round((U.q_fp < 0.1).mean(), 3))
print("spikes per unit:", rng_str(U.n_spikes))
print("unit locations: ", U.location.nunique(), "labels;", (U.layer != "NA").mean().round(3), "of units have a layer")
print("units per parent region:"); print(U.parent.value_counts().to_string())
print("\nAll units in both dandisets:", len(U), "+", len(U473), "=", len(U) + len(U473),
      "| paper: 34,642 sorted, 29,782 after excluding 24 recordings, 24,248 after quality control")

The two dandisets hold 29,782 units together, exactly the paper's count after it excluded 24
recordings and before its unit quality control, so the files hold every sorted unit of the 99
recordings and no label says which units passed. The units table has a two-column `quality`
(ISI violation, false positive rate). The first column runs to hundreds, so it is not the ISI
violation ratio below 0.1 of the Methods in the same units; fewer than half of the units are
below 0.1 on it. Its meaning is not described. The peak-to-trough durations are in ms, as
stated, and agree with the paper's Zenodo metrics files (shown in the link claim).

In [ ]:
# Column descriptions of the trials and electrodes tables, one file per setting
first = {}
for r in recs:
    first.setdefault(r["setting"], r)
for sname, r in first.items():
    f = P.open_asset(r["asset_id"])
    tr = f["intervals/trials"]
    desc = {k: P.dec(tr[k].attrs.get("description", "")) for k in tr.keys() if k != "id"}
    sound = tr["sound"][:] if "sound" in tr else None
    el = f["general/extracellular_ephys/electrodes"]
    eld = sorted({P.dec(el[k].attrs.get("description", "")) for k in el.keys() if k != "id"})
    f.close()
    print(f"{sname} ({r['session']}): trials {len(r['onsets'])} onsets; trial columns: {desc}")
    if sound is not None:
        print(f"   'sound' True in {int(np.sum(sound))} of {len(sound)} trials; 10-kHz tones in the file: "
              f"{0 if r['tones'] is None else len(r['tones'])}")
    print(f"   electrode column descriptions: {eld}")

In the Detection and Context files the trials columns `lick` and `valve` are described as "sound
amplitude"; in the Detection file the `Block` column is described as the trial type (miss, hit,
correct rejection, false alarm), not a block. In the Context file `sound` is False for all 524
trials though the file holds 524 10-kHz tones. The electrode columns (including `location`, `AP`,
`DV`, `ML`) are all described as "my description". The values were read as their names say;
`location` agrees with the paper's Zenodo files for 98.6% of units (zenodo-unit-link). Because the
Detection `Block` column alternates, those recordings are cut into blocks of 50 onsets
(fig2e-category-stability).

In [ ]:
ep = pd.DataFrame([{"session": r["session"], "setting": r["setting"], "epochs": len(r["epochs"]),
                    **pd.Series(r["state"]).value_counts().to_dict(),
                    "lick_periods": len(r["lick"]), "saturation_periods": len(r["sat"]),
                    "sleep_source": ("none (code: ignore)" if r["session"] in P.QUIET_IGNORE else
                                     ("other probe" if r["session"] in P.QUIET_FROM_OTHER else "own"))} for r in recs]).fillna(0)
print(ep.to_string(index=False))
print("\nEpoch length is 3 s for every epoch:", all(np.allclose(np.diff(r["epochs"], axis=1), 3) for r in recs))
print("Onsets per file:", rng_str(ep.epochs), "| active:", rng_str(ep.active))
gaps = np.concatenate([np.diff(r["onsets"]) for r in recs if r["setting"] != "Attention"])
print("Intervals between successive onsets outside the working-memory setting (s):",
      np.percentile(gaps, [1, 5, 50, 95, 99]).round(2), "(Methods: tones at random intervals of 5 to 10 s)")

Licking is marked in 44 files (LickOnset and LickOffset series), saturation of the spike band in
most. Sleep-like periods are detected from each probe's cortical units, or from the probe the
paper's code names for probes with few cortical units, and not for the four hippocampal probes,
as in the code. Outside the working-memory setting the intervals between onsets have a median of
8.5 s (5th to 95th percentile 5.2 to 16.3 s), close to the Methods' 5 to 10 s; 1% are below 4.1 s.
In the working-memory setting the code's onsets are trial starts. The 3-s epochs are 3 s long.

Clock check. The spike times and the stimulus onsets must share a clock: below, the population
rate of PFC units in 25-ms bins around the 10-kHz tone onsets of each file, against the same
with the onsets shifted by 1.5 s (a time with no tone). A response locked to the true onsets, and
not to the shifted ones, shows the clocks agree.

In [ ]:
def tone_contrast(r, shift):
    """Mean rate of all units 0 to 100 ms after the onsets minus 100 to 0 ms before, in Hz per unit."""
    if r["tones"] is None:
        return np.nan
    f = P.open_asset(r["asset_id"])
    st = f["units/spike_times"][:]
    f.close()
    t = r["tones"] + shift
    st.sort()
    post = np.searchsorted(st, t + 0.1) - np.searchsorted(st, t)
    pre = np.searchsorted(st, t) - np.searchsorted(st, t - 0.1)
    return (post.sum() - pre.sum()) / len(t) / 0.1 / len(r["units"])

sample = [r for r in recs if r["tones"] is not None][::6]
cc = pd.DataFrame([{"session": r["session"], "true onsets": tone_contrast(r, 0.0),
                    "shifted 1.5 s": tone_contrast(r, 1.5)} for r in sample])
print("Post minus pre rate (Hz per unit) around 10-kHz tone onsets, every sixth file with tones:")
print(cc.round(3).to_string(index=False))
print("files with a larger change at the true onsets than at the shifted ones:",
      int((cc["true onsets"].abs() > cc["shifted 1.5 s"].abs()).sum()), "of", len(cc))

In every sampled file the rate change at the true onsets is larger than at onsets shifted by
1.5 s (an increase in nine, a decrease in one somatosensory recording), so spikes and onsets
share a clock there. The other files with tones are consistent with it: their units enter the
tone categories of fig5b, whose responses peak tens of ms after the onsets.

## dataset-ki-counts

> Dataset KI holds 24,248 single units from 99 Neuropixels recordings in awake head-fixed mice,
> 52% of them in the PFC and 48% in other brain regions.

Readings of the unit selection (claims file, G3): (a) the dataset's own quality metrics under
the Methods criteria, read as column 0 (ISI violation) below 0.1 or column 1 (false positive
rate) below 0.1; (b) all units. Each also needs at least 12 active epochs with at least 6
spikes (Methods); counting epochs with at least 5 spikes (the "≥4 ISIs" of the Methods' unit
quality control) and over all spontaneous epochs, active or sleep-like, are shown as variants.

In [ ]:
def count_table(D, label):
    rows = []
    for g3, m in [("(a) ISI violation < 0.1", D.q_isi < 0.1), ("(a) false positive rate < 0.1", D.q_fp < 0.1),
                  ("(b) all units", np.ones(len(D), bool))]:
        k = D[m & (D.n_valid >= P.MIN_EPOCHS)]
        rows.append({"data": label, "reading": g3, "units": len(k), "PFC units": int(k.is_pfc.sum()),
                     "PFC %": round(100 * k.is_pfc.mean(), 1)})
    return rows

UK = pd.concat([U, U473], ignore_index=True)
rec_counts = {"001260": len(recs), "001260 + 000473": len(recs) + len(recs473)}
ct = pd.DataFrame(count_table(U, "001260") + count_table(UK, "001260 + 000473"))
print(ct.to_string(index=False))
print("recordings (probes):", rec_counts, "| expected 24,248 ± 2,425 units, 52 ± 5.2 % PFC, 99 recordings, 12,674 ± 1,267 PFC units")

def n_valid_with(recs_, min_spikes, states):
    out = []
    for r in recs_:
        m = np.isin(r["state"], states)
        out.append((r["cnt"][:, m] >= min_spikes).sum(1))
    return np.concatenate(out)

var = []
for lab, rr, D in [("001260", recs, U), ("001260 + 000473", recs + recs473, UK)]:
    for ms in [6, 5]:
        for states, sl in [(["active"], "active"), (["active", "near_sleep", "sleep"], "all spontaneous")]:
            nv = n_valid_with(rr, ms, states)
            k = D[nv >= P.MIN_EPOCHS]
            var.append({"data": lab, "min spikes": ms, "epochs": sl, "units": len(k),
                        "PFC %": round(100 * k.is_pfc.mean(), 1), "PFC units": int(k.is_pfc.sum())})
print("\nVariants of the 12-epoch rule, reading (b):")
print(pd.DataFrame(var).to_string(index=False))

**Result: partially reproduced.** On 001260 alone, 14,564 units meet the 12-epoch rule (reading b), 48.4% in the PFC, from 74 recordings: the dandiset holds 74 of the 99 recordings. With 000473 the recordings are the paper's 99 and the PFC share is 53.6% (expected 52 ± 5.2), but the unit count is 20,354 (expected 24,248 ± 2,425) and the PFC units 10,900 (12,674 ± 1,267). Counting all spontaneous epochs, active or sleep-like, gives 24,898 units and 13,040 PFC units, both within tolerance; the shortfall under the active-epoch rule follows from the larger share of sleep-like epochs found here (extfig1-epoch-counts). Under reading (a) the counts are far lower (4,820 or 13,917).

## fig1h-waveform-classes

> Peak-to-trough durations are bimodal, splitting units into 19,186 ww units (81.1%) and 4,200
> nw units (17.7%), with 862 intermediate units left unclassified.

In [ ]:
wf = []
for lab, D in [("001260", U), ("001260 + 000473", UK)]:
    for g3, m in [("(a) false positive rate < 0.1", D.q_fp < 0.1), ("(b) all units", np.ones(len(D), bool))]:
        k = D[m & (D.n_valid >= P.MIN_EPOCHS)]
        n = k.utype.value_counts()
        tot, cls = len(k), n.get("ww", 0) + n.get("nw", 0)
        wf.append({"data": lab, "reading": g3, "ww": n.get("ww", 0), "nw": n.get("nw", 0), "unclassified": n.get("na", 0),
                   "ww % of all": round(100 * n.get("ww", 0) / tot, 1), "nw % of all": round(100 * n.get("nw", 0) / tot, 1),
                   "ww % of ww+nw": round(100 * n.get("ww", 0) / cls, 1), "nw % of ww+nw": round(100 * n.get("nw", 0) / cls, 1)})
print(pd.DataFrame(wf).to_string(index=False))
print("expected: ww 19,186 ± 1,919, nw 4,200 ± 420, unclassified 862 ± 86, ww 81.1 ± 8.1 %, nw 17.7 ± 1.8 %")
k = UK[UK.n_valid >= P.MIN_EPOCHS]
h, e = np.histogram(k.ptt, np.arange(0, 1.6, 0.02))
print("\nHistogram of peak-to-trough (ms, 0.02-ms bins), 001260 + 000473, reading (b); units beyond 1.6 ms:",
      int((k.ptt >= 1.6).sum()))
left = e[:-1]
m_lo, m_hi, m_mid = left < 0.38, left >= 0.43, (left >= 0.3) & (left < 0.5)
lo_peak, hi_peak = left[m_lo][np.argmax(h[m_lo])], left[m_hi][np.argmax(h[m_hi])]
valley = left[m_mid][np.argmin(h[m_mid])]
print(f"mode below 0.38 ms at {lo_peak:.2f} ms ({h[m_lo].max()} units), mode above 0.43 ms at {hi_peak:.2f} ms ({h[m_hi].max()}),"
      f" minimum between 0.30 and 0.50 ms at {valley:.2f} ms ({h[m_mid].min()})")
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(e[:-1], h, width=0.02, align="edge", color="0.4")
ax.axvline(0.38, color="tab:blue"); ax.axvline(0.43, color="tab:red")
ax.set_xlabel("peak-to-trough (ms)"); ax.set_ylabel("units"); ax.set_title("001260 + 000473, units with >= 12 active epochs")
plt.show()

**Result: partially reproduced.** The fractions match: on 001260 + 000473 (reading b) 78.1% ww and 18.2% nw of all units (81.1% and 18.9% of ww plus nw units), within 81.1 ± 8.1 and 17.7 ± 1.8. The counts are below the expected ones: 15,894 ww (19,186 ± 1,919), 3,698 nw (4,200 ± 420) and 762 unclassified (862 ± 86), as the unit total is (dataset-ki-counts). The histogram has a mode at 0.26 ms and one at 0.64 ms with a minimum at 0.36 ms (258 units against 410 at the lower mode).

## fig1i-nw-higher-metrics

> Narrow-waveform units have significantly higher firing rates, burstiness and memory than
> wide-waveform units.

Mixed model Metric ~ 1 + Unittype + (1|Mouse) per metric (Methods, equation 6), fitted with
statsmodels (REML). The paper's Kenward-Roger F test is not available there; the Wald test of the
unit-type coefficient is used (with thousands of units and 40 mice the degrees of freedom are
large either way). With one comparison per metric Tukey's correction changes nothing.

In [ ]:
cl = U[(U.n_valid >= P.MIN_EPOCHS) & U.utype.isin(["ww", "nw"])].dropna(subset=["log10FR", "B", "M"])
rows = []
for m in ["log10FR", "B", "M"]:
    f, r2 = P.mixed(cl, f'{m} ~ C(utype, Treatment("ww"))')
    rows.append({"metric": m, "nw - ww estimate": f.params.iloc[1], "p": f.pvalues.iloc[1],
                 "median ww": cl[cl.utype == "ww"][m].median(), "median nw": cl[cl.utype == "nw"][m].median(),
                 "n ww": int((cl.utype == "ww").sum()), "n nw": int((cl.utype == "nw").sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("mice:", cl.subject.nunique())
print("\nPer setting (median nw minus median ww):")
print(cl.groupby(["setting", "utype"])[["log10FR", "B", "M"]].median().unstack().pipe(
    lambda d: pd.DataFrame({m: d[(m, "nw")] - d[(m, "ww")] for m in ["log10FR", "B", "M"]})).round(3).to_string())
# variant: the code drops units with log10FR < 0.1
cv = cl[cl.log10FR >= 0.1]
print("\nVariant, units with log10FR >= 0.1 only:")
for m in ["log10FR", "B", "M"]:
    f, _ = P.mixed(cv, f'{m} ~ C(utype, Treatment("ww"))')
    print(f"  {m}: nw - ww = {f.params.iloc[1]:.4f}, p = {f.pvalues.iloc[1]:.2g}, n = {len(cv)}")

**Result: reproduced.** nw units have a higher log10FR (+0.235), B (+0.052) and M (+0.042) than ww units, p below 1e-76 for each, in 11,209 ww and 2,752 nw units from 38 mice; the difference has the same sign in every setting and when units with log10FR < 0.1 are dropped.

## extfig1d-sleep-like-differ

> Firing metrics in sleep-like epochs differ from those in active spontaneous epochs, which is
> why sleep-like epochs are excluded.

ww units with at least 12 active epochs (active sample) and ww units with at least 12 sleep-like
epochs of at least 6 spikes (sleep-like sample), Metric ~ 1 + EpochType + (1|Mouse) (Methods,
equation 7). Sleep-like periods from the Methods' detector; the code's detector as a variant.

In [ ]:
def epoch_type_models(Ux):
    ww_ = Ux[Ux.utype == "ww"]
    a = ww_[ww_.n_valid >= P.MIN_EPOCHS][["subject", "log10FR", "B", "M"]].assign(epoch="active")
    s = ww_[ww_.n_valid_sleep >= P.MIN_EPOCHS][["subject", "log10FR_sleep", "B_sleep", "M_sleep"]]
    s = s.rename(columns=lambda c: c.replace("_sleep", "")).assign(epoch="sleep")
    L = pd.concat([a, s]).dropna()
    rows = []
    for m in ["log10FR", "B", "M"]:
        f, _ = P.mixed(L, f"{m} ~ C(epoch)")
        rows.append({"metric": m, "sleep - active": f.params.iloc[1], "p": f.pvalues.iloc[1],
                     "median active": a[m].median(), "median sleep": s[m].median()})
    return pd.DataFrame(rows), len(a), len(s)

res, na_, ns_ = epoch_type_models(U)
print(f"Methods detector: {na_} ww units with >= 12 active epochs, {ns_} with >= 12 sleep-like epochs "
      "(paper, all of dataset KI: 19,186 and 11,599)")
print(res.to_string(index=False))
recs_code = P.classify_epochs([dict(r) for r in recs], detector="code")
U_code = P.unit_table(recs_code)
res_c, na_c, ns_c = epoch_type_models(U_code)
print(f"\nVariant, the code's detector: {na_c} and {ns_c} units")
print(res_c.to_string(index=False))

**Result: reproduced.** In ww units, sleep-like epochs differ from active epochs in all three metrics: log10FR -0.027 (p = 6.4e-07), B +0.029 (p = 5.3e-64), M -0.013 (p = 2.6e-18), from 11,209 active and 10,381 sleep-like samples. With the code's detector all three also differ (p ≤ 0.0012), though log10FR changes sign (+0.018).

## extfig1-epoch-counts

> Sleep-like episodes occupy a median 6% (IQR 0 to 18%) of recording time; sessions hold a median
> 227 (IQR 172 to 284) spontaneous active epochs and 19 (IQR 1 to 84) sleep-like epochs.

Readings of "recording session": one probe recording (file), or one session of up to three
probes (subject and date; time fraction and epochs from the session's first probe, as all
probes of a session share onsets). The recording time is from the first to the last spike.

In [ ]:
def epoch_table(recs_):
    rows = []
    for r in recs_:
        sl = r["sleep"]
        frac = 0.0 if sl is None or len(sl) == 0 else (np.minimum(sl[:, 1], r["t1"]) - np.maximum(sl[:, 0], r["t0"])).clip(0).sum() / (r["t1"] - r["t0"])
        rows.append({"session": r["session"], "day": r["session"].split("-")[0], "sleep %": 100 * frac,
                     "active epochs": int((r["state"] == "active").sum()), "sleep-like epochs": int((r["state"] == "sleep").sum())})
    return pd.DataFrame(rows)

def iqr_str(x):
    q = np.percentile(x, [25, 50, 75])
    return f"{q[1]:.1f} (IQR {q[0]:.1f} to {q[2]:.1f})"

for det, rr in [("Methods detector", recs), ("code's detector", recs_code)]:
    et = epoch_table(rr)
    per_day = et.groupby("day").first()
    print(det)
    for nm, d in [("per probe recording (n=%d)" % len(et), et), ("per session (n=%d)" % len(per_day), per_day)]:
        print(f"  {nm}: sleep-like time {iqr_str(d['sleep %'])} %, active epochs {iqr_str(d['active epochs'])}, "
              f"sleep-like epochs {iqr_str(d['sleep-like epochs'])}")
    print("  ww units with >= 12 sleep-like epochs of >= 6 spikes:",
          int(((U if det.startswith('M') else U_code).query("utype == 'ww'").n_valid_sleep >= P.MIN_EPOCHS).sum()))
print("expected: 6 ± 0.6 %, 227 ± 22.7 active, 19 ± 1.9 sleep-like epochs, 11,599 ± 1,160 ww units (all of dataset KI)")
et = epoch_table(recs)
print("\nRecordings with more than half of their time sleep-like:", int((et["sleep %"] > 50).sum()), "of", len(et))
print(et.sort_values("sleep %", ascending=False).head(12).round(1).to_string(index=False))
et473 = epoch_table(recs473)
print("\n000473 (25 aversion recordings), Methods detector:", "sleep-like time", iqr_str(et473["sleep %"]),
      "%, active epochs", iqr_str(et473["active epochs"]), ", sleep-like epochs", iqr_str(et473["sleep-like epochs"]))

**Result: not reproduced.** Per probe recording, sleep-like periods take a median 20.2% of the time (IQR 8.4 to 33.0%), against 6% (0 to 18%); recordings hold a median 180.5 active epochs (227 expected) and 125.5 sleep-like epochs (19 expected); 12 of 74 recordings are more than half sleep-like. Per session the values are similar. The code's detector gives 8.7%, 203 and 83.5. The count of ww units with at least 12 sleep-like epochs, 10,381, is just below 11,599 ± 1,160 although it covers 74 of 99 recordings.

## extfig1f-burstiness-memory-uncorrelated

> Across ww units, a linear correlation does not reveal a dependence between burstiness and memory.

A claim of absence, judged on the paper's test: B ~ 1 + M + (1|Mouse) across all ww units
(Methods, equation 8), p < 0.05.

In [ ]:
ww_all = U[(U.utype == "ww") & (U.n_valid >= P.MIN_EPOCHS)].dropna(subset=["log10FR", "B", "M"]).copy()
for form in ["B ~ M", "M ~ B", "B ~ log10FR", "M ~ log10FR"]:
    f, r2 = P.mixed(ww_all, form)
    print(f"{form}: slope {f.params.iloc[1]:.4f}, p = {f.pvalues.iloc[1]:.3g}, marginal R2 = {r2:.4f}, n = {len(ww_all)}")
print("Pearson r(B, M) across ww units:", round(stats.pearsonr(ww_all.B, ww_all.M)[0], 3))

**Result: not reproduced.** By the paper's test burstiness depends on memory: slope -0.19, p = 1.7e-66, across 11,209 ww units. The effect is small: marginal R2 = 0.024 (Pearson r = -0.17).

## The ww and nw SOMs and their categories

Trained on all ww (or nw) units with at least 12 active epochs, all regions and layers (code
`somsettings_carlenSpont.yml`: layerlist all). The two readings of which retrained cluster is
which category (claims file, G2) are shown with each cluster's median metrics.

In [ ]:
fit_ww = P.fit_categories(ww_all, 400, 8)
ww_all["cat_a"], ww_all["cat_b"], ww_all["node"] = fit_ww["cat_a"], fit_ww["cat_b"], fit_ww["node"]
print("ww SOM grid", fit_ww["som"]["shape"], "=", fit_ww["som"]["W"].shape[0], "nodes (code's get_kshape rounds up from 400)")
tab = fit_ww["med"].assign(reading_a=fit_ww["map_a"], reading_b=fit_ww["map_b"],
                           units=np.bincount(fit_ww["lab"][fit_ww["node"]], minlength=8))
print(tab.sort_values("reading_b").round(3).to_string())
print("\nText: 1-3 low rate regular (1 most regular), 4-6 high rate (4 Poissonian, 6 bursty), 7-8 low rate bursty")
print("Clusters given the same number by both readings:", int((fit_ww["map_a"] == fit_ww["map_b"]).sum()), "of 8")
nw_all = U[(U.utype == "nw") & (U.n_valid >= P.MIN_EPOCHS)].dropna(subset=["log10FR", "B", "M"]).copy()
fit_nw = P.fit_categories(nw_all, 220, 5)
nw_all["cat_a"] = fit_nw["cat_a"]
print("\nnw SOM grid", fit_nw["som"]["shape"], "| clusters with the code's numbering (reading a):")
print(fit_nw["med"].assign(reading_a=fit_nw["map_a"], units=np.bincount(fit_nw["lab"][fit_nw["node"]], minlength=5)).round(3).to_string())

## fig2a-som-memory-burstiness

> On the ww SOM, burstiness and firing rate vary along perpendicular gradients, while high memory
> occurs only with medium to low burstiness, so high memory and high burstiness are mutually
> exclusive.

In [ ]:
Wd = fit_ww["som"]["W"] * fit_ww["sd"] + fit_ww["mu"]
pos = fit_ww["som"]["pos"]
ang, gB, gF = P.gradient_angle(pos, Wd[:, 1], Wd[:, 0])
print(f"angle between the B and log10FR gradients: {ang:.1f} degrees (criterion 60 to 120)")
print("node prototypes: log10FR", rng_str(Wd[:, 0]), "| B", rng_str(Wd[:, 1]), "| M", rng_str(Wd[:, 2]))
q75B = np.quantile(Wd[:, 1], 0.75)
for q, lab in [(0.9, "(a) top 10%"), (0.75, "(b) top 25%")]:
    top = Wd[:, 2] >= np.quantile(Wd[:, 2], q)
    print(f"{lab} memory nodes: {top.sum()}, of which in the top quartile of node B: {(Wd[top, 1] >= q75B).sum()} "
          f"(their B {np.round(np.sort(Wd[top, 1])[-3:], 3)}, top-quartile threshold {q75B:.3f})")
print("Spearman r of node M and node B:", round(stats.spearmanr(Wd[:, 2], Wd[:, 1])[0], 3))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, j, nm in zip(axs, range(3), ["log10FR", "B", "M"]):
    sc = ax.scatter(pos[:, 0], pos[:, 1], c=Wd[:, j], marker="h", s=60, cmap="viridis"); ax.set_title(nm); plt.colorbar(sc, ax=ax)
    ax.set_aspect("equal"); ax.axis("off")
plt.show()

**Result: partially reproduced.** The gradients of B and log10FR over the node grid are at 110 degrees, within 60 to 120, so part 1 holds. Part 2 does not: of the top 10% memory nodes 3 of 43 lie in the top quartile of node B, and of the top 25% 23 of 107, where the criterion asks for none.

## fig2b-pfc-regular-territory

> PFC ww units are widely spread on the SOM, with many matching regular-firing (low burstiness)
> nodes and few in anti-memory territory, while units from other regions populate different
> territories.

In [ ]:
pfc, oth = ww_all[ww_all.is_pfc], ww_all[~ww_all.is_pfc]
print(f"PFC share of the ww SOM units: {100 * ww_all.is_pfc.mean():.1f} % ({len(pfc)} of {len(ww_all)}); expected 54 ± 5.4")
mw = stats.mannwhitneyu(pfc.B, oth.B, alternative="two-sided")
print(f"median B: PFC {pfc.B.median():.4f}, other {oth.B.median():.4f}, Mann-Whitney p = {mw.pvalue:.3g}")
tabm = np.array([[(pfc.M < 0).sum(), (pfc.M >= 0).sum()], [(oth.M < 0).sum(), (oth.M >= 0).sum()]])
chi = stats.chi2_contingency(tabm)
print(f"fraction M < 0: PFC {(pfc.M < 0).mean():.3f}, other {(oth.M < 0).mean():.3f}, chi-square p = {chi[1]:.3g}")
print("nodes hit by PFC units:", pfc.node.nunique(), "of", fit_ww["som"]["W"].shape[0], "| by other units:", oth.node.nunique())
print("\nPer setting:")
print(ww_all.groupby(["setting", "is_pfc"]).agg(medB=("B", "median"), fracMneg=("M", lambda x: (x < 0).mean()), n=("B", "size")).round(3).unstack().to_string())

**Result: reproduced.** PFC units are 50.5% of the ww SOM units (54 ± 5.4) and hit all 425 nodes. Compared with ww units of other regions they have a lower median burstiness (-0.055 against -0.023, Mann-Whitney p = 2.4e-32) and fewer anti-memory units (M < 0 in 54.1% against 68.2%, chi-square p = 3.6e-53). The burstiness difference is reversed in the Aversion setting (-0.105 against -0.111).

## fig2e-category-stability

> About 57% of ww units keep their category from one block of about 50 spontaneous epochs to the
> next, far above the 18% expected by chance from the marginal distributions.

Blocks: where the trials table's `Block` column marks runs of trials (Aversion, Context, Opto
and part of Passive), each run is a block, runs of more than 75 epochs split in two; elsewhere
consecutive blocks of 50 onsets (assumption, as the files do not say which recordings had an
inherent block structure). Block metrics need at least 12 active epochs of at least 6 spikes
(Methods); the code's minimum of 3 is a variant. Stability does not depend on which cluster is
given which number, so G2 does not apply.

In [ ]:
print("Block structure per file:", pd.Series([r["blockmode"] for r in recs]).value_counts().to_dict(),
      "| blocks per file:", rng_str([len(np.unique(r["blocks"])) for r in recs]))
bc = P.block_categories(recs, ww_all, fit_ww)
print("unit-blocks:", len(bc), "| units with at least one block:", bc.groupby(["session", "unit_id"]).ngroups,
      "| epochs per analysed block (median, IQR):", np.percentile(bc.n_epochs, [50, 25, 75]))
Ms = P.transitions(bc, "cat_b", 8)
st_t = pd.DataFrame([{"transition": t, "units": int(M.sum()), "stability": P.stability(M)[0], "chance": P.stability(M)[1]}
                     for t, M in Ms.items()])
print(st_t.round(3).to_string(index=False))
Msum = sum(Ms.values())
print(f"(a) mean over transitions: stability {st_t.stability.mean():.3f}, chance {st_t.chance.mean():.3f}")
print(f"(b) pooled matrix: stability {P.stability(Msum)[0]:.3f}, chance {P.stability(Msum)[1]:.3f}")
print("expected 0.57 ± 0.057 and 0.18 ± 0.018")
bc3 = P.block_categories(recs, ww_all, fit_ww, min_valid=3)
Ms3 = P.transitions(bc3, "cat_b", 8)
s3 = [P.stability(M) for M in Ms3.values()]
print(f"\nVariant, blocks with >= 3 valid epochs: mean {np.mean([s[0] for s in s3]):.3f} (chance {np.mean([s[1] for s in s3]):.3f}),"
      f" pooled {P.stability(sum(Ms3.values()))[0]:.3f}")
bcs = bc.merge(ww_all[["session", "unit_id", "setting"]], on=["session", "unit_id"])
print("\nPooled stability by setting:")
for sname, g in bcs.groupby("setting"):
    Mg = sum(P.transitions(g, "cat_b", 8).values())
    if np.sum(Mg) > 0:
        print(f"  {sname}: {P.stability(Mg)[0]:.3f} (chance {P.stability(Mg)[1]:.3f}, {int(Mg.sum())} transitions)")

**Result: reproduced.** Units keep their category from one block to the next in 57.3% of transitions on average over 13 transitions (59.4% for the pooled matrix), against 18.4% expected from the marginals; expected 57 ± 5.7 and 18 ± 1.8. Each transition lies between 46.2% and 61.1%. With the code's minimum of 3 epochs per block, stability is 49.0% (chance 14.7%).

## fig2f-low-rate-less-stable

> Block-wise category assignment is less stable for low-rate categories (1 to 3, 7 and 8) than
> for high-rate categories, and low-rate regular categories 1 to 3 mostly transition among
> themselves, as do the low-rate bursty categories 7 and 8.

In [ ]:
def fig2f_parts(bc_, cat):
    M = sum(P.transitions(bc_, cat, 8).values())
    Pm = M / M.sum(1, keepdims=True)
    X = P.coincidence(M)
    low, high = [0, 1, 2, 6, 7], [3, 4, 5]
    d, dx = np.diag(Pm), np.diag(X)
    part1 = d[low].mean() < d[high].mean()
    part1x = dx[low].mean() < dx[high].mean()
    part2 = []
    for grp in ([0, 1, 2], [6, 7]):
        for i in grp:
            others = [j for j in grp if j != i]
            out = [j for j in range(8) if j not in grp]
            part2.append(Pm[i, others].sum() > Pm[i, out].max())
    return Pm, X, d, dx, part1, part1x, part2

for cat in ["cat_a", "cat_b"]:
    Pm, X, d, dx, p1, p1x, p2 = fig2f_parts(bc, cat)
    print(f"G2 reading {cat[-1]}: diagonal of P by category 1..8: {np.round(d, 3)}")
    print(f"   mean diagonal low-rate (1-3,7,8) {d[[0,1,2,6,7]].mean():.3f} vs high-rate (4-6) {d[[3,4,5]].mean():.3f} -> part 1 {p1};"
          f" coincidence diagonal {np.round(dx, 3)} -> {p1x}")
    print(f"   part 2 (within-group probability above any single outside category), categories 1,2,3,7,8: {p2}")
    print(pd.DataFrame(Pm, index=range(1, 9), columns=range(1, 9)).round(2).to_string())

**Result: blocked (methods underspecified).** The outcome depends on which retrained cluster is given which category number (G2). Under the text-matched numbering (b) low-rate categories are less stable (mean diagonal 0.430 against 0.691, the coincidence diagonal agrees), but part 2 holds for 2 of 5 categories: partially reproduced. Under the code's numbering (a) neither part holds (0.613 against 0.387). The claims file blocks a claim whose outcome differs between the two.

## Region enrichment of ww categories

Deep-layer units for cortical regions and all units elsewhere, against all such units
(the code's refall), PFC subregions and the code's parent regions, regions with more than 20
units, 1,000 shuffles within each behavioral setting. G1: (a) five groups, Opto merged into
Passive (code); (b) six settings. Both G2 readings.

In [ ]:
ww_all["setting5"] = ww_all.setting.replace({"Opto": "Passive"})
sel = ww_all[(~ww_all.cortical) | ww_all.deep]
cnt = sel.region.value_counts()
regs = cnt[(cnt > 20) & (cnt.index != "other")].index
print("units:", len(sel), "| regions with more than 20 units:", len(regs))
print(cnt[regs].to_string())
ENR = {}
for g1, gcol in [("a", "setting5"), ("b", "setting")]:
    for g2 in ["a", "b"]:
        E, Pv, Xc = P.escore(sel.region.values, sel["cat_" + g2].values, sel[gcol].values, 1000, seed=11,
                             pops=list(regs) + ["other"], attrs=range(1, 9))
        ENR[(g1, g2)] = (E.loc[regs], Pv.loc[regs])
show = ["ILA", "PL", "ACAd", "ACAv", "ORBm", "ORBvl", "ORBl", "MOs", "AIv", "AId", "FRP", "TH", "HPF"]
for key, (E, Pv) in ENR.items():
    print(f"\nG1 {key[0]}, G2 {key[1]}: E-scores (* P < 0.05)")
    print(E.loc[[r for r in show if r in E.index]].round(1).astype(str).where(Pv.loc[[r for r in show if r in E.index]] >= 0.05,
          E.loc[[r for r in show if r in E.index]].round(1).astype(str) + "*").to_string())

## fig2g-pfc-regular-enrichment

> Most PFC subregions (ILA, PL, ACAd, ACAv, ORBm and ORBvl) are enriched in low-rate,
> regular-firing ww units (categories 1 to 3).

In [ ]:
six = ["ILA", "PL", "ACAd", "ACAv", "ORBm", "ORBvl"]
rows = []
for (g1, g2), (E, Pv) in ENR.items():
    ra = (E.loc[six, [1, 2, 3]].sum(1) > 0)
    rb = ((E.loc[six, [1, 2, 3]] > 0) & (Pv.loc[six, [1, 2, 3]] < 0.05)).any(axis=1)
    rows.append({"G1": g1, "G2": g2, **{f"(a) {r}": bool(ra[r]) for r in six}, "(a) n": int(ra.sum()),
                 **{f"(b) {r}": bool(rb[r]) for r in six}, "(b) n": int(rb.sum())})
res2g = pd.DataFrame(rows)
print(res2g.to_string(index=False))
for (g1, g2), (E, Pv) in ENR.items():
    print(f"G1 {g1} G2 {g2}: sum of E-scores of categories 1-3:", E.loc[six, [1, 2, 3]].sum(1).round(1).to_dict())

**Result: blocked (methods underspecified).** Under the text-matched numbering (G2 b) all six subregions are enriched in categories 1 to 3 under both readings and both groupings (sums of E-scores 3.9 to 16.2, each with a significant positive E-score): reproduced. Under the code's numbering (G2 a) ACAd and ACAv are not (sums -4.8 to -1.4): 4 of 6, partially reproduced. The claims file blocks a claim whose outcome differs under G2.

## fig2g-ila-pl-category1

> ILA and PL are both strongly and significantly enriched in the most regular-firing category 1.

In [ ]:
for (g1, g2), (E, Pv) in ENR.items():
    print(f"G1 {g1} G2 {g2}: category 1 E-score ILA {E.loc['ILA', 1]:.2f} (P {Pv.loc['ILA', 1]:.3f}), "
          f"PL {E.loc['PL', 1]:.2f} (P {Pv.loc['PL', 1]:.3f})")

**Result: blocked (methods underspecified).** Under G2 b, ILA and PL are significantly enriched in category 1 (E = 8.7 to 9.1 and 6.0, P < 0.001); under G2 a, ILA is depleted (E = -3.0 to -3.1) and PL not enriched. The outcome differs under G2.

## fig2g-bursty-orbl-mos-aiv

> ORBl, MOs and AIv are significantly enriched in bursty category 8 units, setting them apart
> from the activity-defined prefrontal entity.

In [ ]:
for (g1, g2), (E, Pv) in ENR.items():
    print(f"G1 {g1} G2 {g2}: category 8 E-score " + ", ".join(
        f"{r} {E.loc[r, 8]:.2f} (P {Pv.loc[r, 8]:.3f})" if r in E.index else f"{r} not analysed" for r in ["ORBl", "MOs", "AIv"]))

**Result: blocked (methods underspecified).** Under G2 b, ORBl (E = 3.6 to 3.7) and MOs (4.0 to 4.1) are significantly enriched in category 8 and AIv is not (1.4 to 1.6, P 0.07 to 0.11): partially reproduced. Under G2 a, none of the three is enriched (MOs is depleted). The outcome differs under G2.

## fig2g-th-hpf-bursty

> Thalamus and hippocampal formation are strongly and significantly enriched in bursty units
> (categories 6 to 8), unlike the PFC.

In [ ]:
for (g1, g2), (E, Pv) in ENR.items():
    for r in ["TH", "HPF"]:
        sig = [(E.loc[r, c] > 0) & (Pv.loc[r, c] < 0.05) for c in [6, 7, 8]]
        print(f"G1 {g1} G2 {g2} {r}: E-scores 6-8 {E.loc[r, [6, 7, 8]].round(2).tolist()}, P {Pv.loc[r, [6, 7, 8]].round(3).tolist()};"
              f" (a) all three {all(sig)}, (b) at least one {any(sig)}")

**Result: blocked (methods underspecified).** Under both G2 readings TH and HPF each have one significant positive E-score among categories 6 to 8 (TH 7.1 to 7.3, HPF 3.8 to 4.0), so the claim's reading (b), at least one, holds; reading (a), all three, holds for neither. The claims file blocks the claim when its two readings disagree.

## extfig3f-layer-enrichment

> Superficial cortical layers (L2/3) are enriched in bursty ww units, while deep layers (L5 to 6)
> are enriched in regular-firing ww units.

All cortical ww units with a layer, groups = region × setting (Methods, Group-bias control).

In [ ]:
cx = ww_all[ww_all.cortical & (ww_all.layer != "NA")].copy()
cx["L"] = cx.layer.str.replace(r"^6[ab]$", "6", regex=True)
print("layer labels:", cx.L.value_counts().to_dict())
for g1, gcol in [("a", "setting5"), ("b", "setting")]:
    grp = cx.region + "|" + cx[gcol]
    for g2 in ["a", "b"]:
        E, Pv, Xc = P.escore(cx.L.values, cx["cat_" + g2].values, grp.values, 1000, seed=12, attrs=range(1, 9))
        sigpos = (E > 0) & (Pv < 0.05)
        ok23 = sigpos.loc["2/3", [6, 7, 8]].any() if "2/3" in E.index else False
        ok5 = sigpos.loc["5", [1, 2, 3]].any(); ok6 = sigpos.loc["6", [1, 2, 3]].any()
        print(f"\nG1 {g1}, G2 {g2}: L2/3 enriched in 6-8: {ok23}; L5 in 1-3: {ok5}; L6 in 1-3: {ok6}")
        print(E.round(1).astype(str).where(Pv >= 0.05, E.round(1).astype(str) + "*").to_string())

**Result: blocked (methods underspecified).** Under G2 b, L2/3 is enriched in bursty category 6 and 8 units (E = 3.5 to 3.7 and 6.4 to 7.0) and L5 (category 2, E = 1.9 to 2.2) and L6 (category 1, E = 1.8 to 2.0) in regular categories: reproduced. Under G2 a, L5 has no significant positive E-score in categories 1 to 3. The outcome differs under G2.

## extfig5g-nw-enrichment

> Among nw units, ILA, PL and ACAd/v are significantly enriched in high-rate, regular-firing units
> with high memory, and ORBl in high-rate, bursty units.

The regular high-memory nw category is the one with median log10FR above the median over all nw
units and the lowest median B; the bursty one has median log10FR above that median and the
highest median B (claims file, by category medians).

In [ ]:
med_nw = fit_nw["med"].copy()
med_nw["cat"] = fit_nw["map_a"]
med_nw = med_nw.set_index("cat").sort_index()
hi_rate = med_nw[med_nw.log10FR > nw_all.log10FR.median()]
reg_cat, burst_cat = hi_rate.B.idxmin(), hi_rate.B.idxmax()
print(med_nw.round(3).to_string())
print(f"median log10FR of nw units {nw_all.log10FR.median():.3f}; regular high-memory category {reg_cat} "
      f"(M {med_nw.loc[reg_cat, 'M']:.3f}), bursty category {burst_cat}")
nw_all["setting5"] = nw_all.setting.replace({"Opto": "Passive"})
seln = nw_all[(~nw_all.cortical) | nw_all.deep]
cntn = seln.region.value_counts()
regs_n = cntn[(cntn > 20) & (cntn.index != "other")].index
print("nw regions with more than 20 units:", len(regs_n), cntn[regs_n].to_dict())
ENRN = {}
for g1, gcol in [("a", "setting5"), ("b", "setting")]:
    E, Pv, _ = P.escore(seln.region.values, seln.cat_a.values, seln[gcol].values, 1000, seed=13,
                        pops=list(regs_n) + ["other"], attrs=range(1, 6))
    ENRN[g1] = (E.loc[regs_n], Pv.loc[regs_n])
    sp = (E > 0) & (Pv < 0.05)
    g = lambda r, c: bool(sp.loc[r, c]) if r in sp.index else False
    p1 = g("ILA", reg_cat) and g("PL", reg_cat) and (g("ACAd", reg_cat) or g("ACAv", reg_cat))
    p2 = g("ORBl", burst_cat)
    print(f"\nG1 {g1}: part 1 (ILA, PL, ACAd or ACAv in category {reg_cat}) {p1}; part 2 (ORBl in {burst_cat}) {p2}")
    print(E.loc[regs_n].round(1).astype(str).where(Pv.loc[regs_n] >= 0.05, E.loc[regs_n].round(1).astype(str) + "*").to_string())

**Result: not reproduced.** The nw categories as numbered by the code put the high-rate, lowest-burstiness category at 5 (median M 0.018) and the high-rate bursty one at 1. ILA (E = -0.5 to -0.6) and PL (-2.0 to -2.1, depleted) are not enriched in category 5, and ORBl is not enriched in category 1 (E = 1.5, not significant), under both groupings.

## fig2j-nw-ww-correspondence

> Brain (sub)regions enriched in low-rate, regular-firing ww units are enriched in high-rate,
> regular-firing nw units, and regions enriched in low-rate bursty ww units are enriched in
> high-rate bursty nw units.

In [ ]:
for g1 in ["a", "b"]:
    En = ENRN[g1][0]
    for g2 in ["a", "b"]:
        Ew = ENR[(g1, g2)][0]
        common = [r for r in En.index if r in Ew.index]
        R = pd.DataFrame([[stats.pearsonr(Ew.loc[common, cw], En.loc[common, cn])[0] for cn in range(1, 6)] for cw in range(1, 9)],
                         index=[f"ww{c}" for c in range(1, 9)], columns=[f"nw{c}" for c in range(1, 6)])
        r_reg = R.loc[["ww1", "ww2", "ww3"], f"nw{reg_cat}"].mean()
        r_bur = R.loc[["ww7", "ww8"], f"nw{burst_cat}"].mean()
        print(f"G1 {g1}, G2 {g2}: {len(common)} regions (paper 9); mean r ww1-3 vs nw{reg_cat} = {r_reg:.3f}; "
              f"mean r ww7-8 vs nw{burst_cat} = {r_bur:.3f}")
        if g2 == "b":
            print(R.round(2).to_string())

**Result: partially reproduced.** Across 21 regions with more than 20 nw units (9 in the paper), the mean correlation between the E-scores of ww categories 1 to 3 and of the regular nw category is -0.077 to -0.001, not positive, under every reading. The mean correlation between ww categories 7 and 8 and the bursty nw category is positive but small, 0.040 to 0.115.

## Harris hierarchy (Fig. 3)

The Harris et al. 2019 scores as the code reads them. Populations are the cortical areas with a
score (deep layers), all other units form one rest population (code
`harris_vs_enrichment.py`); E-scores grouped by setting (Methods), and without groups as the
code's Fig. 3 script does (variant). Areas with more than 20 units.

In [ ]:
harris = P.harris_table()
print("Harris cortical areas in the table:", len(harris))
hsel = ww_all.copy()
hsel["areaH"] = np.where(hsel.area.isin(harris.index) & hsel.deep, hsel.area, "NA")
hc = hsel.areaH.value_counts()
areasH = [a for a in hc.index if a != "NA" and hc[a] > 20]
print("areas with a score and more than 20 deep ww units:", len(areasH), "(paper 17)")
print(pd.DataFrame({"units": hc[areasH], "harris": harris[areasH]}).sort_values("harris").round(3).to_string())
HAR = {}
for lab, grp in [("G1 a", hsel.setting.replace({"Opto": "Passive"}).values), ("G1 b", hsel.setting.values),
                 ("no groups", np.zeros(len(hsel)))]:
    for g2 in ["a", "b"]:
        E, Pv, _ = P.escore(hsel.areaH.values, hsel["cat_" + g2].values, grp, 1000, seed=14, pops=areasH + ["NA"], attrs=range(1, 9))
        E = E.loc[areasH]
        HAR[(lab, g2)] = (E, Pv.loc[areasH])
        rr = [stats.pearsonr(harris[areasH], E[c]) for c in range(1, 9)]
        print(f"{lab}, G2 {g2}: r by category " + ", ".join(f"{c}: {r[0]:.2f} (p {r[1]:.3f})" for c, r in zip(range(1, 9), rr)))

## fig3a-harris-regular-positive

> In dataset KI, enrichment of cortical (sub)regions in low-rate, regular-firing units
> (categories 1 to 3, category 2 shown) correlates positively with the Harris cortical hierarchy
> score, driven by the split between low-hierarchy sensory regions and high-hierarchy PFC
> subregions.

In [ ]:
def harris_r(E, cats, areas=None):
    a = areasH if areas is None else areas
    return {c: stats.pearsonr(harris[a], E.loc[a, c]) for c in cats}

for key, (E, Pv) in HAR.items():
    rr = harris_r(E, [1, 2, 3])
    ok = rr[2][0] > 0 and rr[2][1] < 0.05 and rr[1][0] > 0 and rr[3][0] > 0
    part = rr[2][0] > 0 and rr[2][1] < 0.05
    print(f"{key}: r1 {rr[1][0]:.2f} (p {rr[1][1]:.3f}), r2 {rr[2][0]:.2f} (p {rr[2][1]:.3f}), r3 {rr[3][0]:.2f} (p {rr[3][1]:.3f})"
          f" -> all {ok}, category 2 alone {part}")
E, _ = HAR[("G1 a", "b")]
pf = [a for a in areasH if a in P.PFC_SUBREGIONS]
print("\nG1 a, G2 b: mean category 2 E-score, PFC areas", round(E.loc[pf, 2].mean(), 2), "| other areas",
      round(E.loc[[a for a in areasH if a not in pf], 2].mean(), 2), "| Harris score PFC", round(harris[pf].mean(), 3),
      "other", round(harris[[a for a in areasH if a not in pf]].mean(), 3))

**Result: blocked (methods underspecified),** because the claim depends on fig2g-pfc-regular-enrichment, which is blocked (protocol: a claim that depends on a blocked claim is blocked for the same reason). As computed it would be not reproduced: across 15 cortical areas with a Harris score and more than 20 deep ww units (17 in the paper), category 2 enrichment does not correlate significantly with hierarchy under any reading: r = 0.30 to 0.40 (p 0.14 to 0.28) under G2 b, and r = -0.17 to 0.05 under G2 a. Under G2 b categories 1 and 3 are positive as well (r 0.28 to 0.50, p ≥ 0.057). PFC areas have higher category 2 E-scores (0.28) and Harris scores (0.26) than the other areas (-2.51 and -0.16).

## fig3c-harris-bursty-negative

> Cortical hierarchy correlates even more strongly, and negatively, with enrichment in bursty,
> low-memory units (categories 6 to 8), in both datasets.

In [ ]:
for key, (E, Pv) in HAR.items():
    rb, rr = harris_r(E, [6, 7, 8]), harris_r(E, [1, 2, 3])
    neg = all(rb[c][0] < 0 for c in [6, 7, 8]) and any(rb[c][1] < 0.05 for c in [6, 7, 8])
    stronger = max(abs(rb[c][0]) for c in [6, 7, 8]) > max(abs(rr[c][0]) for c in [1, 2, 3])
    print(f"{key}: r6 {rb[6][0]:.2f} (p {rb[6][1]:.3f}), r7 {rb[7][0]:.2f} (p {rb[7][1]:.3f}), r8 {rb[8][0]:.2f} (p {rb[8][1]:.3f});"
          f" all negative, one significant: {neg}; largest |r| 6-8 above 1-3: {stronger}")

**Result: blocked (methods underspecified).** Under G2 b, categories 6 to 8 all correlate negatively with hierarchy (r = -0.57 to -0.53 for 6, p 0.027 to 0.040), more strongly than any of 1 to 3: reproduced for dataset KI. Under G2 a, category 6 correlates positively (r = 0.30 to 0.40). The outcome differs under G2. The IBL Passive part was not tested.

## supptable5-no-hierarchy-within-pfc

> Restricted to PFC subregions, enrichment in unit categories shows no relationship to cortical
> hierarchy.

A claim of absence: Pearson correlation across the PFC subregions among the areas above, per
category, p < 0.05, no correction.

In [ ]:
print("PFC subregions with a Harris score and more than 20 units:", pf)
for key, (E, Pv) in HAR.items():
    rr = harris_r(E, range(1, 9), pf)
    sig = [c for c in range(1, 9) if rr[c][1] < 0.05]
    print(f"{key}: " + ", ".join(f"{c}: {rr[c][0]:.2f} (p {rr[c][1]:.3f})" for c in range(1, 9)) + f" -> significant: {sig}")

**Result: blocked (methods underspecified),** because the claim depends on fig3a-harris-regular-positive, which is blocked. As computed the absence holds: within the 11 PFC subregions with a Harris score, no category's E-scores correlate with hierarchy at p < 0.05 under any reading (|r| ≤ 0.35, p ≥ 0.28).

## zenodo-unit-link

> Link claim (the registry's): the paper's per-unit metrics files on Zenodo can be matched to the
> dataset's units, and the matched records agree on what both describe.

Records are matched by recording id and unit id. Agreement is checked on the location (area
and layer), the peak-to-trough duration, and the ww/nw label against the Methods thresholds
applied to the dataset's `Peak_to_Trough`. The files' quality flags are compared with the
paper's count of units kept.

In [ ]:
Z = P.paper_unit_table()
print("Zenodo metrics files:", Z.session.nunique(), "recordings,", len(Z), "units")
L = U.merge(Z, on=["session", "unit_id"], how="left", indicator=True)
print("dataset units matched:", int((L._merge == "both").sum()), "of", len(L), "| recordings matched:",
      L[L._merge == "both"].session.nunique(), "of", U.session.nunique())
Lm = L[L._merge == "both"].copy()
zal = Lm.z_location.map(P.split_area_layer)
Lm["z_area"], Lm["z_layer"] = [a for a, _ in zal], [l for _, l in zal]
print("same location label:", round((Lm.location == Lm.z_location).mean(), 4), "| same area:", round((Lm.area == Lm.z_area).mean(), 4))
print("peak-to-trough equal within 0.001 ms:", round((np.abs(Lm.ptt - Lm.z_p2t_ms) < 1e-3).mean(), 4))
lab = np.where(Lm.z_ww == 1, "ww", np.where(Lm.z_nw == 1, "nw", "na"))
print("ww/nw label agreement:", round((lab == Lm.utype).mean(), 4)); print(pd.crosstab(Lm.utype, lab, rownames=["dataset thresholds"], colnames=["Zenodo label"]).to_string())
diff = Lm[Lm.location != Lm.z_location]
print(f"\nunits whose location differs: {len(diff)}; examples:")
print(diff[["session", "unit_id", "location", "z_location"]].head(10).to_string(index=False))
print("by recording (units with a different location):"); print(diff.session.value_counts().head(10).to_string())
print("\nZenodo quality flags among the matched units: quality == 1:", int((Lm.z_quality == 1).sum()),
      "| waveform quality == 1:", int((Lm.z_wquality == 1).sum()), "| both:", int(((Lm.z_quality == 1) & (Lm.z_wquality == 1)).sum()))
both_ok = (Lm.z_quality == 1) & (Lm.z_wquality == 1)
print("units with both flags and >= 12 active epochs:", int((both_ok & (Lm.n_valid >= P.MIN_EPOCHS)).sum()),
      "| with both flags and >= 12 spontaneous epochs (active or sleep-like):",
      int((both_ok & (n_valid_with(recs, 6, ['active', 'near_sleep', 'sleep']) [L._merge.values == 'both'] >= P.MIN_EPOCHS)).sum()))
print("flatmap positions: PFC units with U > 0:", int(((Lm.U > 0) & Lm.is_pfc).sum()), "of", int(Lm.is_pfc.sum()), "PFC units")
Z473 = Z[Z.session.isin(U473.session)]
print("\n000473 recordings in the Zenodo files:", Z473.session.nunique(), "| units:", len(Z473), "of", len(U473))

**Result: reproduced.** Every one of the 20,374 units of 001260 has a record in the Zenodo metrics files, matched by recording and unit id. The records agree on the location label for 98.6% of units (283 differ, 207 of them in one recording) and on the ww/nw label for 97.0%; peak-to-trough agrees within 0.001 ms for 95.1%. The paper's own quality flags pass 19,808 units; with the 12-epoch rule 14,256 remain.

## fig4b-dataroi-parcellation

> The PFC flatmap is parcellated into 42 dataROIs that respect subregion boundaries, each holding
> about 200 deep-layer ww units.

Deep-layer PFC ww units with at least 12 active epochs and a flatmap position, assigned to a
subregion by the subregion polygon that contains them (code `flatmap_tessellation.py`); the
dataset's location label is shown as a check. Size bounds 160 to 240 (Methods), 180 to 220 as a
variant (code).

In [ ]:
regpolys = P.flatmap_polygons("flatmap_PFCregions.h5")
uvmap = Z.set_index(["session", "unit_id"])[["U", "V"]]
def attach_uv(D):
    D = D.join(uvmap, on=["session", "unit_id"])
    return D

ww_all = attach_uv(ww_all)
fm = ww_all[ww_all.is_pfc & ww_all.deep & (ww_all.U > 0)].copy()
fm["polyreg"] = P.in_polygons(fm[["U", "V"]].values, regpolys)
print("deep PFC ww units with a flatmap position:", len(fm), "| inside a subregion polygon:", int(fm.polyreg.notna().sum()),
      "| polygon agrees with the location label:", round((fm.polyreg == fm.area).mean(), 3))
fm = fm[fm.polyreg.notna()]
print("units per subregion:", fm.polyreg.value_counts().to_dict())
fm["droi"] = P.parcellate(fm[["U", "V"]].values, fm.polyreg.values, bounds=(160, 240))
fm["droi_code"] = P.parcellate(fm[["U", "V"]].values, fm.polyreg.values, bounds=(180, 220))
sz = fm.droi.value_counts()
print(f"\n001260: dataROIs (160-240) {fm.droi.nunique()}, (180-220) {fm.droi_code.nunique()}; units per dataROI {rng_str(sz)}")
# all of dataset KI (001260 + 000473): counts only
ww473 = U473[(U473.utype == "ww") & (U473.n_valid >= P.MIN_EPOCHS) & U473.is_pfc & U473.deep]
ww473 = attach_uv(ww473)
k473 = ww473[ww473.U > 0].copy()
k473["polyreg"] = P.in_polygons(k473[["U", "V"]].values, regpolys)
kk = pd.concat([fm[["U", "V", "polyreg"]], k473[k473.polyreg.notna()][["U", "V", "polyreg"]]])
nk = P.parcellate(kk[["U", "V"]].values, kk.polyreg.values, bounds=(160, 240))
nk2 = P.parcellate(kk[["U", "V"]].values, kk.polyreg.values, bounds=(180, 220))
print(f"001260 + 000473: {len(kk)} units, dataROIs (160-240) {len(set(nk))}, (180-220) {len(set(nk2))}; per subregion",
      kk.polyreg.value_counts().to_dict())
paper_drois = P.flatmap_polygons("flatmap_PFC_ntesselated_obeyRegions_res200.h5")
print("the paper's own dataROI tessellation on Zenodo has", len(paper_drois), "polygons; expected 42 ± 4")

**Result: reproduced.** On dataset KI (001260 + 000473), 7,867 deep PFC ww units with flatmap positions give 38 dataROIs under either size bound, within 42 ± 4; the paper's own tessellation on Zenodo has 42. On 001260 alone, which holds 4,906 such units, the same rule gives 22 dataROIs. 87.7% of units fall in the subregion polygon that matches their location label.

## fig4cd-dataroi-enrichment

> Low-rate, regular-firing categories 1 and 3 are significantly enriched in dataROIs in ILA and PL,
> while low-rate, bursty categories 7 and 8 are enriched in dataROIs in MOs and ORBl, the ORBl
> enrichment pinpointed to its most ventral dataROI.

Reference (a) all deep PFC ww units of the parcellation; (b) all units as for regions (the
dataROIs plus one rest population of every other unit of the region analysis). Groups by
setting (G1 a and b), both G2 readings, dataROIs from the 160-240 parcellation.

In [ ]:
def droi_escores(fm_, roicol, ref, g1, g2, seed):
    if ref == "a":
        pops, cats = fm_[roicol].values, fm_["cat_" + g2].values
        grp = fm_.setting.replace({"Opto": "Passive"}).values if g1 == "a" else fm_.setting.values
        plist = sorted(pd.unique(pops))
    else:
        rest = sel[~sel.set_index(["session", "unit_id"]).index.isin(fm_.set_index(["session", "unit_id"]).index)]
        both = pd.concat([fm_.assign(pp=fm_[roicol]), rest.assign(pp="rest")])
        pops, cats = both.pp.values, both["cat_" + g2].values
        grp = both.setting.replace({"Opto": "Passive"}).values if g1 == "a" else both.setting.values
        plist = sorted(pd.unique(fm_[roicol])) + ["rest"]
    E, Pv, _ = P.escore(pops, cats, grp, 1000, seed=seed, pops=plist, attrs=range(1, 9))
    keep = [p for p in plist if p != "rest"]
    return E.loc[keep], Pv.loc[keep]

DROI = {}
for ref in ["a", "b"]:
    for g1 in ["a", "b"]:
        for g2 in ["a", "b"]:
            E, Pv = droi_escores(fm, "droi", ref, g1, g2, seed=21)
            DROI[(ref, g1, g2)] = (E, Pv)
            sp = (E > 0) & (Pv < 0.05)
            sub = lambda r, cs: any(sp.loc[i, cs].any() for i in E.index if i.startswith(r + "-"))
            ok = sub("ILA", [1, 3]) and sub("PL", [1, 3]) and sub("MOs", [7, 8]) and sub("ORBl", [7, 8])
            orbl = [i for i in E.index if i.startswith("ORBl-")]
            best = E.loc[orbl, [7, 8]].sum(1).idxmax() if orbl else None
            print(f"reference {ref}, G1 {g1}, G2 {g2}: ILA 1/3 {sub('ILA', [1, 3])}, PL 1/3 {sub('PL', [1, 3])}, "
                  f"MOs 7/8 {sub('MOs', [7, 8])}, ORBl 7/8 {sub('ORBl', [7, 8])} -> {ok}; ORBl dataROI with the highest 7+8 E-score: {best}")
E, Pv = DROI[("a", "a", "b")]
print("\nreference a, G1 a, G2 b (* P < 0.05):")
print(E.round(1).astype(str).where(Pv >= 0.05, E.round(1).astype(str) + "*").to_string())

**Result: reproduced.** On the 22 dataROIs of 001260, at least one dataROI in ILA and in PL is significantly enriched in category 1 or 3, and at least one in MOs and in ORBl in category 7 or 8, under both references, both groupings and both G2 readings. ORBl is a single dataROI here (380 units), so whether the enrichment is in its most ventral part is not tested.

## fig4e-activity-modules

> Clustering dataROIs by enrichment profile gives activity modules; ILA forms a homogeneous
> ILA-specific module bordering a similar module in adjacent PL and ACAd, while other modules
> cross cytoarchitectural boundaries and occur in several places.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
for key, (E, Pv) in DROI.items():
    Zs = (E - E.mean(0)) / E.std(0)
    mod = pd.Series(fcluster(linkage(Zs.values, "ward"), 5, "maxclust"), index=E.index)
    subr = mod.index.str.split("-").str[0]
    ila = mod[subr == "ILA"]
    one = ila.nunique() == 1
    m_ila = ila.iloc[0] if one else None
    rd_a = one and (subr[mod == m_ila] == "ILA").all()
    rd_b = one and (subr[mod == m_ila] == "ILA").mean() > 0.5
    multi = sum(len(set(subr[mod == m])) > 1 for m in mod.unique() if m != m_ila)
    print(f"ref {key[0]} G1 {key[1]} G2 {key[2]}: ILA dataROIs {len(ila)} in one module {one}; ILA-only (a) {rd_a}, "
          f"mostly ILA (b) {rd_b}; other modules spanning subregions {multi}")
    if key == ("a", "a", "b"):
        print(pd.crosstab(mod.values, subr, rownames=["module"], colnames=["subregion"]).to_string())

**Result: partially reproduced.** The two ILA dataROIs of 001260 fall in one module under every reading, and three or four other modules hold dataROIs of more than one subregion. The ILA module is not ILA-specific: it also holds two PL, one MOs and one ORBm dataROI, so it is neither ILA-only (a) nor mostly ILA (b). Both readings agree that this part fails.

## GaoROIs and the intra-PFC hierarchy (Fig. 4g,h, Extended Data Fig. 7n)

Units are assigned to the 60 GaoROI polygons (Zenodo `flatmap_PFCrois.h5`); GaoROIs with at
least 20 deep ww units (Extended Data Fig. 7i; more than 20 as a variant) and a hierarchy score.
E-scores as for dataROIs (both references, G1, G2), Pearson correlation with the Gao hierarchy
index.

In [ ]:
gpolys = P.flatmap_polygons("flatmap_PFCrois.h5")
gh = gao_h = P.gao_hierarchy()
fm["groi"] = P.in_polygons(fm[["U", "V"]].values, gpolys)
fm["groi"] = fm.groi.astype("float").astype("Int64")
gcount = fm.groi.value_counts()
print("deep PFC ww units in a GaoROI:", int(fm.groi.notna().sum()), "of", len(fm), "| GaoROIs with units:", len(gcount),
      "| with >= 20:", int((gcount >= 20).sum()), "| with >= 20 and a score:", int(((gcount >= 20) & gcount.index.isin(gh.dropna().index)).sum()),
      "| with > 20 and a score:", int(((gcount > 20) & gcount.index.isin(gh.dropna().index)).sum()), "(paper 30)")
GAO = {}
for nmin_lab, keep in [(">= 20", gcount[gcount >= 20].index), ("> 20", gcount[gcount > 20].index)]:
    keep = [int(k) for k in keep if k in gh.dropna().index]
    fg = fm[fm.groi.isin(keep)].copy()
    fg["groi_s"] = fg.groi.astype(int).astype(str)
    for ref in ["a", "b"]:
        for g1 in ["a", "b"]:
            for g2 in ["a", "b"]:
                E, Pv = droi_escores(fg, "groi_s", ref, g1, g2, seed=31)
                E.index = E.index.astype(int); Pv.index = Pv.index.astype(int)
                GAO[(nmin_lab, ref, g1, g2)] = (E, Pv)
for key, (E, Pv) in GAO.items():
    h = gh[E.index]
    r1, r6 = stats.pearsonr(h, E[1]), stats.pearsonr(h, E[6])
    print(f"n {key[0]}, ref {key[1]}, G1 {key[2]}, G2 {key[3]}: {len(E)} GaoROIs; category 1 r = {r1[0]:.2f} (p {r1[1]:.3f}),"
          f" category 6 r = {r6[0]:.2f} (p {r6[1]:.3f})")

## fig4g-gao-category1

> Across GaoROIs, enrichment in low-rate, regular-firing category 1 units correlates positively
> with the intra-PFC hierarchy score of Gao et al.

In [ ]:
for key, (E, Pv) in GAO.items():
    r = stats.pearsonr(gh[E.index], E[1])
    print(f"n {key[0]}, ref {key[1]}, G1 {key[2]}, G2 {key[3]}: r = {r[0]:.3f}, p = {r[1]:.4f}, n = {len(E)} -> {r[0] > 0 and r[1] < 0.05}")

**Result: not reproduced.** Across 29 GaoROIs (30 in the paper), category 1 enrichment correlates positively but not significantly with the Gao hierarchy index under every reading: r = 0.20 to 0.33, p = 0.086 to 0.31.

## extfig7n-gao-category6

> Across GaoROIs, enrichment in high-rate, bursty category 6 units correlates negatively with
> intra-PFC hierarchy.

In [ ]:
for key, (E, Pv) in GAO.items():
    r = stats.pearsonr(gh[E.index], E[6])
    print(f"n {key[0]}, ref {key[1]}, G1 {key[2]}, G2 {key[3]}: r = {r[0]:.3f}, p = {r[1]:.4f} -> {r[0] < 0 and r[1] < 0.05}")

**Result: blocked (methods underspecified).** Under G2 b, category 6 enrichment correlates negatively with hierarchy (r = -0.62 to -0.54, p ≤ 0.0023, 29 GaoROIs): reproduced. Under G2 a, r = 0.03 to 0.11. The outcome differs under G2.

## fig4g-pl-orbl-extremes

> GaoROIs 19, 20 and 41 in PL have exceptionally high hierarchy scores and the strongest category 1
> enrichment, while GaoROI 23 in ventral ORBl has the lowest hierarchy score and the strongest
> category 1 depletion.

In [ ]:
print("Gao table: highest hierarchy scores", gh.dropna().sort_values().index[-4:].tolist()[::-1], "| lowest", gh.dropna().sort_values().index[:2].tolist())
for key, (E, Pv) in GAO.items():
    s = E[1].sort_values()
    top3, low = s.index[-3:].tolist()[::-1], s.index[0]
    n_top = len({19, 20, 41} & set(top3))
    print(f"n {key[0]}, ref {key[1]}, G1 {key[2]}, G2 {key[3]}: top three category 1 {top3}, lowest {low};"
          f" 19/20/41 in top three: {n_top}; 23 analysed: {23 in E.index}; 23 lowest: {low == 23}")

**Result: blocked (methods underspecified).** The Gao table itself gives GaoROI 23 the lowest score, but 5, 24, 3 and 2 the highest, not 19, 20 and 41. Under G2 b, GaoROIs 20 and 41 are among the three with the highest category 1 E-scores (with 56) and the lowest is 6, not 23: partially reproduced. Under G2 a, none of 19, 20 and 41 is in the top three, and 23 is the lowest in 6 of 8 readings. The outcome differs between readings.

## Tone responses (Fig. 5b, 5j)

Tone response traces as in the Methods (pfcki.tone_traces): 10-kHz tones only (the
working-memory recordings have none and are left out), epochs with licking, sleep-like periods
or saturation dropped, units below 0.1 Hz overall or before the tone dropped, PCA over ww and nw
units pooled, 8 components. A SOM of 400 nodes on ww units, trained on (a) deep-layer ww units of
all regions (Results) or (b) all ww units (code); Ward into 8 categories numbered by the peak
latency of their mean trace, the category with the lowest mean peak last (Methods: "the flat or
negative category last").

In [ ]:
from concurrent.futures import ThreadPoolExecutor
trecs = [r for r in recs if r["tones"] is not None]
with ThreadPoolExecutor(8) as ex:
    TT = list(ex.map(P.tone_traces, trecs))
T = pd.concat([pd.DataFrame({"session": r["session"], "unit_id": r["units"].unit_id.values, "rate_all": o["rate_all"],
                             "rate_pre": o["rate_pre"], "n_tone_epochs": o["n_tones"]}) for r, o in zip(trecs, TT)], ignore_index=True)
X = np.vstack([o["trace"] for o in TT])
T = T.merge(U[["session", "unit_id", "utype", "is_pfc", "deep", "area", "region", "setting", "subject", "cortical"]], on=["session", "unit_id"])
print("files with 10-kHz tones:", len(trecs), "| tone response epochs per file:", rng_str([o["n_tones"] for o in TT]),
      "(paper: 188, IQR 86 to 228)")
okT = ((T.rate_all >= 0.1) & (T.rate_pre >= 0.1) & np.isfinite(X).all(1) & T.utype.isin(["ww", "nw"])).values
print("units:", len(T), "| kept (rates >= 0.1 Hz, ww or nw):", int(okT.sum()))
T, X = T[okT].reset_index(drop=True), X[okT]
Xc = X - X.mean(0)
w, V = np.linalg.eigh(np.cov(Xc.T))
o = np.argsort(w)[::-1][:8]
pcs = V[:, o].T
print(f"variance explained by 8 components: {w[o].sum() / w.sum():.3f} (Methods: over 95%)")
Sc = X @ pcs.T
tt = np.arange(X.shape[1]) * P.PSTH_BIN
TONE = {}
for rd, m in [("a", ((T.utype == "ww") & ((~T.cortical) | T.deep)).values), ("b", (T.utype == "ww").values)]:
    Z0 = Sc[m]
    Zs = (Z0 - Z0.mean(0)) / Z0.std(0)
    som_t = P.train_som(Zs, 400)
    lab = P.ward_clusters(som_t["W"], 8)
    cl = lab[P.bmus(Zs, som_t["W"])]
    mt = np.array([X[m][cl == c].mean(0) for c in range(8)])
    peak, lat = mt.max(1), tt[mt.argmax(1)]
    flat = int(np.argmin(peak))
    order = [c for c in np.argsort(lat) if c != flat] + [flat]
    num = np.zeros(8, int)
    for i, c in enumerate(order):
        num[c] = i + 1
    D = T[m].copy()
    D["tcat"] = num[cl]
    TONE[rd] = D
    pf = D[D.is_pfc & D.deep]
    print(f"\nreading {rd}: {m.sum()} ww units trained, {len(pf)} PFC deep ww units (paper 7,184 for all of dataset KI)")
    print(pd.DataFrame({"category": num, "mean peak (z)": peak.round(2), "peak latency (ms)": (lat * 1000).round(1),
                        "units": np.bincount(cl, minlength=8), "PFC deep ww": [int((pf.tcat == num[c]).sum()) for c in range(8)]}
                       ).sort_values("category").to_string(index=False))
fig, axs = plt.subplots(1, 2, figsize=(11, 3))
for ax, rd in zip(axs, ["a", "b"]):
    D = TONE[rd]
    for c in range(1, 9):
        sel_ = (T.set_index(["session", "unit_id"]).index.isin(D[D.tcat == c].set_index(["session", "unit_id"]).index))
        ax.plot(tt * 1000, X[sel_].mean(0), label=str(c))
    ax.set_title(f"reading {rd}: mean trace per category"); ax.set_xlabel("ms after tone"); ax.set_ylabel("z")
axs[1].legend(ncol=4, fontsize=7)
plt.show()

## fig5b-tone-categories

> Deep-layer PFC ww tone responses cluster into eight categories ordered by peak delay; most
> units have flat or negative responses (category 8), categories 1 to 4 have sharp peaks with
> consistent onsets, and categories 5 to 7 have variable onsets and persistent responses.

ZETA test (pfcki.zeta_unit) on the PFC deep ww units of categories 1 to 7 under either reading;
the spread of onsets is the IQR over units with P < 0.01.

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
# all PFC deep ww units with a tone trace (needed for fig5e as well)
need = TONE["b"][TONE["b"].is_pfc & TONE["b"].deep][["session", "unit_id"]].drop_duplicates()
by_s = {r["session"]: r for r in trecs}
args, keys = [], []
for s, g in need.groupby("session"):
    a = P.zeta_for(by_s[s], g.unit_id.values, seed=5)
    args += a
    keys += [(s, u) for u in g.unit_id.values]
t0 = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    zres = list(ex.map(P.zeta_unit, args, chunksize=8))
ZT = pd.DataFrame(keys, columns=["session", "unit_id"]).assign(zeta_p=[z[0] for z in zres], onset=[z[1] for z in zres])
print(f"ZETA on {len(ZT)} units in {time.time() - t0:.0f} s; failed or empty: {int(ZT.zeta_p.isna().sum())}")
for rd in "ab":
    D = TONE[rd][TONE[rd].is_pfc & TONE[rd].deep].merge(ZT, on=["session", "unit_id"], how="left")
    cnt = D.tcat.value_counts().sort_index()
    largest = cnt.idxmax() == 8
    rows = []
    for c in range(1, 8):
        g = D[(D.tcat == c) & (D.zeta_p < 0.01)]
        q = np.percentile(g.onset.dropna(), [25, 50, 75]) * 1000 if len(g.onset.dropna()) >= 3 else [np.nan] * 3
        rows.append({"category": c, "units": int((D.tcat == c).sum()), "ZETA P < 0.01": len(g),
                     "onset median (ms)": round(q[1], 1), "onset IQR (ms)": round(q[2] - q[0], 1)})
    R = pd.DataFrame(rows)
    iqr = R.set_index("category")["onset IQR (ms)"]
    part2 = iqr.loc[1:4].max() < iqr.loc[5:7].min()
    print(f"\nreading {rd}: category 8 is the largest: {largest} ({cnt.get(8, 0)} of {len(D)} units)")
    print(R.to_string(index=False))
    print(f"largest IQR in 1-4: {iqr.loc[1:4].max():.1f} ms, smallest IQR in 5-7: {iqr.loc[5:7].min():.1f} ms -> part 2 {part2}")

**Result: reproduced.** Under both training readings the flat or negative category 8 is the largest among the 5,232 PFC deep ww units (3,114 and 3,788). The across-unit IQR of ZETA onsets of responsive units is 3.9 to 35.8 ms in categories 1 to 4, and 90.7 to 284.4 ms in categories 5 to 7. The mean traces of categories 1 to 7 peak at 18.5 to 578.5 ms after onset, in order of category number. Recordings keep a median 80 tone epochs (188 in the paper), because more epochs are sleep-like here.

## fig5cd-tone-maps-cross-subregions

> Tone-response enrichment maps of the PFC do not follow cytoarchitectural boundaries; the
> earliest-responding units (category 1) are enriched in anterior ACAd and ORBvl and parts of PL,
> and tone response modules reveal heterogeneity within subregions.

dataROIs of fig4b; units are the PFC deep ww units with a tone category; E-scores of tone
category per dataROI grouped by setting, reference (a) these units, (b) all ww units with a
tone category in the region analysis (deep cortical and all non-cortical); Ward of the
standardized profiles into 4 modules (Extended Data Fig. 8e). Both tone readings.

In [ ]:
for rd in "ab":
    D = TONE[rd]
    ft = fm.merge(D[["session", "unit_id", "tcat"]], on=["session", "unit_id"])
    reg_all = D[(~D.cortical) | D.deep]
    for ref in "ab":
        for g1 in "ab":
            if ref == "a":
                pops, cats, sett, plist = ft.droi.values, ft.tcat.values, ft.setting, sorted(ft.droi.unique())
            else:
                rest = reg_all[~reg_all.set_index(["session", "unit_id"]).index.isin(ft.set_index(["session", "unit_id"]).index)]
                both = pd.concat([ft.assign(pp=ft.droi), rest.assign(pp="rest")])
                pops, cats, sett, plist = both.pp.values, both.tcat.values, both.setting, sorted(ft.droi.unique()) + ["rest"]
            grp = sett.replace({"Opto": "Passive"}).values if g1 == "a" else sett.values
            E, Pv, _ = P.escore(pops, cats, grp, 1000, seed=51, pops=plist, attrs=range(1, 9))
            E, Pv = E.loc[sorted(ft.droi.unique())], Pv.loc[sorted(ft.droi.unique())]
            sp1 = (E[1] > 0) & (Pv[1] < 0.05)
            subr = E.index.str.split("-").str[0]
            hit = {r: bool(sp1[subr == r].any()) for r in ["ACAd", "ORBvl", "PL"]}
            Zs = (E - E.mean(0)) / E.std(0).replace(0, 1)
            mod = pd.Series(fcluster(linkage(Zs.fillna(0).values, "ward"), 4, "maxclust"), index=E.index)
            split = [r for r in pd.unique(subr) if mod[subr == r].nunique() > 1]
            print(f"tone {rd}, ref {ref}, G1 {g1}: {len(ft)} units; category 1 enriched dataROI in ACAd/ORBvl/PL {hit};"
                  f" subregions whose dataROIs fall in more than one module: {split} -> {all(hit.values()) and len(split) >= 2}")
            if rd == "a" and ref == "a" and g1 == "a":
                print(E.round(1).astype(str).where(Pv >= 0.05, E.round(1).astype(str) + "*").assign(module=mod.values).to_string())

**Result: partially reproduced.** Under every reading, ACAd and PL each have a dataROI significantly enriched in tone category 1, but ORBvl has none. The dataROIs of three to five subregions fall in more than one tone module, so tone maps do not follow subregion boundaries.

## fig5e-tone-responsive-patches

> Tone-responsive ww units (ZETA test) are enriched in spatial patches distributed across the PFC.

Responsive at ZETA P < 0.01 (Methods); dataROI E-scores of responsive against nonresponsive PFC
deep ww units with a tone trace, grouped by setting (G1 a, b), references as above.

In [ ]:
fz = fm.merge(ZT, on=["session", "unit_id"])
fz = fz[fz.zeta_p.notna()].copy()
fz["resp"] = np.where(fz.zeta_p < 0.01, "responsive", "non")
print(f"PFC deep ww units with a flatmap position and a ZETA result: {len(fz)}; responsive: {int((fz.resp == 'responsive').sum())}"
      f" ({100 * (fz.resp == 'responsive').mean():.1f} %)")
ZETAE = {}
for g1 in "ab":
    grp = fz.setting.replace({"Opto": "Passive"}).values if g1 == "a" else fz.setting.values
    E, Pv, _ = P.escore(fz.droi.values, fz.resp.values, grp, 1000, seed=61, attrs=["responsive", "non"])
    ZETAE[g1] = (E, Pv)
    e, p = E["responsive"], Pv["responsive"]
    subr = e.index.str.split("-").str[0]
    enr, dep = (e > 0) & (p < 0.05), (e < 0) & (p < 0.05)
    both = [r for r in pd.unique(subr) if enr[subr == r].any() and dep[subr == r].any()]
    print(f"G1 {g1}: enriched dataROIs {list(e.index[enr])} in {subr[enr].nunique()} subregions; depleted {list(e.index[dep])};"
          f" subregions with both: {both} -> {subr[enr].nunique() >= 3 and len(both) >= 1}")
print(pd.DataFrame({"E": ZETAE["a"][0]["responsive"].round(2), "P": ZETAE["a"][1]["responsive"].round(3),
                    "units": fz.droi.value_counts()}).to_string())

**Result: reproduced.** 1,746 of 3,839 PFC deep ww units are tone-responsive (45.5%). Significantly enriched dataROIs lie in 5 (G1 a) or 6 (G1 b) subregions, and ACAd and MOs (and PL under G1 a) each hold both an enriched and a depleted dataROI.

## extfig9e-tone-responsive-no-hierarchy

> Enrichment in tone-responsive units does not correlate with intra-PFC hierarchy across GaoROIs.

A claim of absence: Pearson correlation of the GaoROI E-scores of ZETA-responsive units with the
Gao hierarchy index, GaoROIs with at least 20 units and a score, p < 0.05.

In [ ]:
gzc = fz.groi.value_counts()
for nmin_lab, keepg in [(">= 20", gzc[gzc >= 20].index), ("> 20", gzc[gzc > 20].index)]:
    keepg = [int(k) for k in keepg if k in gh.dropna().index]
    fg = fz[fz.groi.isin(keepg)]
    for g1 in "ab":
        grp = fg.setting.replace({"Opto": "Passive"}).values if g1 == "a" else fg.setting.values
        E, Pv, _ = P.escore(fg.groi.astype(int).values, fg.resp.values, grp, 1000, seed=71, attrs=["responsive", "non"])
        r = stats.pearsonr(gh[E.index], E["responsive"])
        print(f"n {nmin_lab}, G1 {g1}: {len(E)} GaoROIs (paper 29), r = {r[0]:.3f}, p = {r[1]:.3f} -> absence {r[1] >= 0.05}")

**Result: reproduced.** Across 25 GaoROIs (29 in the paper), enrichment in tone-responsive units does not correlate with the Gao hierarchy: r = 0.046 (p = 0.83) or -0.002 (p = 0.99).

## fig5i-map-similarity

> No pair of PFC maps is strongly congruent, but the choice-tuning map is most similar to the
> spontaneous category 1 map, the feedback-tuning map to the category 6 map, and the
> tone-responsiveness map to the category 8 map.

Dataset part only (the choice and feedback maps need dataset IBL, not tested). Each map is
rasterized on the flatmap (pixels of 2 flatmap units, a standard choice), each pixel taking the
E-score of its dataROI; cosine similarity between the tone-responsiveness map (fig5e, G1 a and b)
and each spontaneous category map (fig4cd, reference a, same G1), both G2 readings.

In [ ]:
pix, plab = P.raster_rois(fm[["U", "V"]].values, fm.polyreg.values, fm.droi.values, regpolys, step=2.0)
print(f"pixels: {len(pix)} (paper 311,563 at its own resolution), dataROIs covered: {len(set(plab))}")
for g1 in "ab":
    tmap = ZETAE[g1][0]["responsive"].reindex(plab).values
    for g2 in "ab":
        E = DROI[("a", g1, g2)][0]
        cos = {c: float(np.dot(tmap, E[c].reindex(plab).values) / np.linalg.norm(tmap) / np.linalg.norm(E[c].reindex(plab).values)) for c in range(1, 9)}
        best = max(cos, key=cos.get)
        print(f"G1 {g1}, G2 {g2}: cosine of the tone-responsiveness map with categories 1-8: "
              + ", ".join(f"{c}: {v:.2f}" for c, v in cos.items()) + f" -> highest {best}")

**Result: blocked (too large),** because the claim depends on fig5g-tuned-unit-counts (dataset IBL), which is blocked. The dataset part, as computed, is not reproduced: the tone-responsiveness map is most similar to spontaneous category 1 (G2 a, cosine 0.34 or 0.46) or 5 (G2 b, the same cluster), never to category 8 (cosine -0.02 to 0.07), under both groupings. No pair is strongly congruent: the largest cosine is 0.46. The choice and feedback parts need dataset IBL and were not tested.

## fig5j-tone-responsive-high-rate

> At the single-unit level, tone-responsive units (tone categories 1 to 7) are more often than
> expected high-rate spontaneous units (categories 4 to 6), and units without clear positive tone
> responses (category 8) tend to have low spontaneous rates (categories 1 to 3, 7 and 8).

Mutual enrichment: spontaneous category as population, tone category as attribute, PFC deep ww
units with both, 1,000 shuffles grouped by setting (G1 a, b); the code's grouping by region and
setting as a variant. Both tone readings and both G2 readings.

In [ ]:
for rd in "ab":
    D = TONE[rd][TONE[rd].is_pfc & TONE[rd].deep].merge(ww_all[["session", "unit_id", "cat_a", "cat_b"]], on=["session", "unit_id"])
    print(f"tone reading {rd}: PFC deep ww units with both categories: {len(D)} (expected 6,681 ± 668 for all of dataset KI)")
    for g2 in "ab":
        for glab, grp in [("G1 a", D.setting.replace({"Opto": "Passive"}).values), ("G1 b", D.setting.values),
                          ("region x setting", (D.area + "|" + D.setting).values)]:
            E, Pv, _ = P.escore(D["cat_" + g2].values, D.tcat.values, grp, 1000, seed=41, pops=range(1, 9), attrs=range(1, 9))
            s17 = E[list(range(1, 8))].sum(1)
            sig = ((E[list(range(1, 8))] > 0) & (Pv[list(range(1, 8))] < 0.05)).loc[[4, 5, 6]].any(axis=1).any()
            p1 = (s17.loc[[4, 5, 6]] > 0).all() and sig
            n8 = int((E.loc[[1, 2, 3, 7, 8], 8] > 0).sum())
            print(f"  G2 {g2}, {glab}: summed E 1-7 in spont 4/5/6 {s17.loc[[4, 5, 6]].round(1).tolist()} (one significant {sig});"
                  f" tone-8 E-score positive in {n8} of spont 1,2,3,7,8 {E.loc[[1, 2, 3, 7, 8], 8].round(1).tolist()} -> {p1 and n8 >= 4}")

**Result: not reproduced.** Tone category 8 has a positive E-score in only 2 or 3 of the five low-rate spontaneous categories under every reading (four required). The summed E-score of tone categories 1 to 7 is positive in spontaneous categories 4, 5 and 6 only under G2 b with tone reading a; under G2 a it is negative in category 6 (-10.4 to -8.4). 3,921 PFC deep ww units have both categories (6,681 ± 668 expected for all of dataset KI).

## methods-settings-no-difference (variant only)

> Firing metric distributions do not differ significantly across behavioral settings.

The paper names no test; the claim is blocked. As a variant only, a mixed model
Metric ~ 1 + Setting + (1|Mouse) per metric for ww units, with the size of the setting effects.

In [ ]:
for m in ["log10FR", "B", "M"]:
    f, r2 = P.mixed(ww_all, f'{m} ~ C(setting, Treatment("Passive"))')
    eff = f.params[[k for k in f.params.index if k.startswith("C(")]]
    pv = f.pvalues[eff.index]
    print(f"{m}: setting effects vs Passive " + ", ".join(f"{k.split('T.')[-1][:-1]} {v:.3f} (p {p:.2g})" for (k, v), p in zip(eff.items(), pv)),
          f"| marginal R2 {r2:.4f}")
print("medians by setting:"); print(ww_all.groupby("setting")[["log10FR", "B", "M"]].median().round(3).to_string())

## Summary

| Claim | Outcome |
|---|---|
| dataset-ki-counts | partially reproduced (99 recordings and 53.6% PFC with 000473; 20,354 units) |
| fig1h-waveform-classes | partially reproduced (fractions match, counts low) |
| fig1i-nw-higher-metrics | reproduced |
| extfig1d-sleep-like-differ | reproduced |
| extfig1-epoch-counts | not reproduced (20% sleep-like time, 6% expected) |
| extfig1f-burstiness-memory-uncorrelated | not reproduced (p = 1.7e-66, marginal R2 0.024) |
| fig2a-som-memory-burstiness | partially reproduced |
| fig2b-pfc-regular-territory | reproduced |
| fig2e-category-stability | reproduced (57.3%, chance 18.4%) |
| fig2f-low-rate-less-stable | blocked: methods underspecified (G2) |
| extfig3f-layer-enrichment | blocked: methods underspecified (G2) |
| fig2g-pfc-regular-enrichment | blocked: methods underspecified (G2) |
| fig2g-ila-pl-category1 | blocked: methods underspecified (G2) |
| fig2g-bursty-orbl-mos-aiv | blocked: methods underspecified (G2) |
| fig2g-th-hpf-bursty | blocked: methods underspecified (the claim's two readings) |
| fig2i-ibl-replication | blocked: too large (dataset IBL, not attempted) |
| extfig5g-nw-enrichment | not reproduced |
| fig2j-nw-ww-correspondence | partially reproduced |
| fig3a-harris-regular-positive | blocked: methods underspecified (depends on fig2g-pfc-regular-enrichment; not reproduced as computed) |
| fig3b-harris-regular-positive-ibl | blocked: too large (dataset IBL) |
| fig3c-harris-bursty-negative | blocked: methods underspecified (G2) |
| supptable5-no-hierarchy-within-pfc | blocked: methods underspecified (depends on fig3a; the absence holds as computed) |
| zenodo-unit-link (link claim) | reproduced |
| fig4b-dataroi-parcellation | reproduced (38 dataROIs with 000473; 22 on 001260 alone) |
| fig4cd-dataroi-enrichment | reproduced |
| fig4e-activity-modules | partially reproduced |
| fig4g-gao-category1 | not reproduced |
| extfig7n-gao-category6 | blocked: methods underspecified (G2) |
| fig4g-pl-orbl-extremes | blocked: methods underspecified |
| fig5b-tone-categories | reproduced |
| fig5cd-tone-maps-cross-subregions | partially reproduced |
| fig5e-tone-responsive-patches | reproduced |
| fig5g-tuned-unit-counts | blocked: too large (dataset IBL) |
| fig5g-tuning-maps | blocked: too large (dataset IBL) |
| fig5h-choice-gao-hierarchy | blocked: too large (dataset IBL) |
| extfig9e-tone-responsive-no-hierarchy | reproduced |
| fig5i-map-similarity | blocked: too large (depends on fig5g-tuned-unit-counts; the dataset part is not reproduced as computed) |
| fig5j-tone-responsive-high-rate | not reproduced |
| fig5k-task-tuning-category4 | blocked: too large (dataset IBL) |
| methods-settings-no-difference | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The analyses run on 001260, which holds 74 of the
99 recordings; 000473 is added only for the counts and the number of dataROIs. Units are not
filtered by the paper's quality flags, which the files lack (the flags on Zenodo pass 97% of
units). Sleep-like periods follow the Methods and flag far more time than the paper reports, so
fewer active epochs and units enter every analysis. The SOM, the clustering and the E-scores are
re-implemented from the Methods and the code; the SOM grid has 425 nodes, as the code's shape rule
gives. Mixed models are fitted with statsmodels and tested by Wald tests, not Kenward-Roger F
tests. The retrained clusters cannot be tied to the paper's category numbers, and the two
readings of the claims file (G2) often disagree. Blocks for the stability analysis are taken
from the trials table's `Block` column or cut into 50 onsets. The ZETA onset is computed from
zetapy's rate as the first half-peak crossing.

**What was found about the dataset.** The two dandisets together hold exactly the 29,782 units
the paper had before its unit quality control, and the 66 mice; 001260 alone holds 74
recordings, 41 mice and 20,374 units. It carries no label of which units the paper kept, its
`quality` columns are undescribed, and its unit locations, flatmap positions and ww/nw labels
need the paper's Zenodo files to be matched to the paper's. The other 25 recordings are in
000473, whose metadata names another paper.